***Mileage Prediction using Deep Learning***

In [59]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder , LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [60]:
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\car_price_prediction_.csv")
df

,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang
...,...,...,...,...,...,...,...,...,...,...
2495,2496,Audi,2020,2.4,Petrol,Automatic,22650,Like New,61384.10,Q5
2496,2497,Audi,2001,5.7,Hybrid,Manual,77701,Like New,24710.35,A3
2497,2498,Ford,2021,1.1,Hybrid,Manual,272827,Like New,29902.45,Fiesta
2498,2499,Audi,2002,4.5,Diesel,Manual,229164,Like New,46085.67,Q5


In [61]:
df["car_age"] = 2026 - df["Year"]
df["mpy"] = (df["Mileage"] /(df["car_age"] +1))

In [62]:
df = df.drop(columns=["Car ID"])

In [63]:
df.head()

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model,car_age,mpy
0,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X,10,10439.272727
1,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series,8,15910.000000
2,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4,13,12971.500000
3,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y,15,4292.625000
4,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang,17,12389.388889


In [64]:
le = LabelEncoder()
df["Brand"] = le.fit_transform(df["Brand"])
df["Fuel Type"] = le.fit_transform(df["Fuel Type"])
df["Transmission"] = le.fit_transform(df["Transmission"])
df["Condition"] = le.fit_transform(df["Condition"])
df["Model"] = le.fit_transform(df["Model"])

In [102]:
X = df.drop(["Mileage"] , axis=1)
y = df["Mileage"]

cat = X.select_dtypes(include="object").columns
num = X.select_dtypes(exclude="object").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=40
)

In [103]:
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop = "first"), cat),
        ("num" , StandardScaler(),num)
    ]
)
pre

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [104]:
# Drop non-numeric columns
X_train_numeric = X_train.select_dtypes(exclude=['object'])
X_test_numeric = X_test.select_dtypes(exclude=['object'])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.fit_transform(X_test_numeric)


In [105]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [106]:
model = Sequential()

model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1,activation = "linear"))   # no activation for regression

model.compile(
    optimizer=Adam(learning_rate = 0.001),
    loss='mse',      # correct loss
    metrics=['mae']
)

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [107]:
model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=20,
    validation_data=(X_test, y_test)
)

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 30064637952.0000 - mae: 149141.9844 - val_loss: 30480744448.0000 - val_mae: 152116.4375
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 30004701184.0000 - mae: 148949.2500 - val_loss: 30322581504.0000 - val_mae: 151613.8125
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 29582417920.0000 - mae: 147623.1562 - val_loss: 29531316224.0000 - val_mae: 149125.9688
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 28143939584.0000 - mae: 143129.7656 - val_loss: 27341246464.0000 - val_mae: 142186.7188
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 24910108672.0000 - mae: 132967.0781 - val_loss: 23094407168.0000 - val_mae: 128235.5938
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 19690422272.0000 - mae: 115787.2500 - val_loss: 17202952192.0000 - val_mae: 107897.5312
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 13801910272.0000 - mae: 94384.6875 - val_loss: 11627

In [108]:
y_pred = model.predict(X_test)
y_pred2 = model.predict(X_train)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [109]:
from sklearn.metrics import r2_score

acc = r2_score(y_test , y_pred)
print(acc)

0.5342221260070801


In [83]:
df.head()

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model,car_age,mpy
0,5,2016,2.3,3,1,114832,1,26613.92,19,10,10439.272727
1,1,2018,4.4,1,1,143190,2,14679.61,1,8,15910.000000
2,0,2013,4.5,1,1,181601,1,44402.61,3,13,12971.500000
3,5,2011,4.1,0,0,68682,1,86374.33,20,15,4292.625000
4,2,2009,2.6,0,1,223009,0,73577.10,21,17,12389.388889


In [91]:
live = pd.DataFrame({
    'Brand': ["Tesla"], 
    'Year': [2016], 
    'Engine Size': [2.3],
    'Fuel Type': ["Petrol"],
    'Transmission': ["Manual"],
    'Condition': ['New'],   # fixed
    'Price' : [26613.56],
    'Model': ["Model X"],
    'car_age' :[10],
    'mpy' : [10439.27]
})

In [92]:
live = pd.get_dummies(live)

live = live.reindex(columns=X.columns, fill_value=0)

live = scaler.transform(live)

pred = model.predict(live)

print("Predicted Mileage:", pred[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
Predicted Mileage: [108546.24]
